# Benchmark Runner
This notebook allows you to select benchmark groups from the `benchmarks` directory, run them, and analyze the results.

In [ ]:
import os
import subprocess
import re
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pathlib import Path
from typing import List, Dict, Optional

EXE_PATH = os.path.join('build-bidirectional', 'bidirectional', 'Release', 'REI.exe')
BENCHMARKS_DIR = 'benchmarks'
DEFAULT_COSTS = [1, 1, 1, 1, 1]
MAX_COST = 500

def parse_output(output_text):
    out_idx = output_text.find("==== Output ===")
    if out_idx != -1:
        output_text = output_text[out_idx:]
        
    if "failed to find any solution" in output_text:
        return {
            "RE": "not_found",
            "Cost": 0,
            "# REs": 0,
            "ICSize": 0,
            "Time (s)": 0.0
        }
    
    running_time_match = re.search(r'Running Time: ([\d.]+) s', output_text)
    running_time_s = float(running_time_match.group(1)) if running_time_match else 0.0

    final_cost_match = re.search(r'Cost: (\d+)', output_text)
    final_cost = int(final_cost_match.group(1)) if final_cost_match else 0

    num_res_match = re.search(r'All REs: (\d+)', output_text, re.MULTILINE)
    if not num_res_match:
        num_res_match = re.search(r'^REs: (\d+)', output_text, re.MULTILINE)
    num_res = int(num_res_match.group(1)) if num_res_match else 0

    ic_size_match = re.search(r'Size of IC: (\d+)', output_text)
    ic_size = int(ic_size_match.group(1)) if ic_size_match else 0

    re_match = re.search(r'RE: "(.*)"', output_text)
    final_re = re_match.group(1) if re_match else "not_found"

    return {
        "RE": final_re,
        "Cost": final_cost,
        "# REs": num_res,
        "ICSize": ic_size,
        "Time (s)": running_time_s
    }

def get_sort_val(file_path):
    import re
    parts = file_path.replace('\\', '/').split('/')
    sort_key = []
    for part in parts:
        sub_parts = []
        for chunk in re.split(r'(\d+)', part):
            if chunk:
                if chunk.isdigit():
                    sub_parts.append((0, int(chunk)))
                else:
                    sub_parts.append((1, chunk))
        sort_key.append(tuple(sub_parts))
    return tuple(sort_key)

def get_benchmark_groups():
    groups = []
    for root, dirs, files in os.walk(BENCHMARKS_DIR):
        if any(f.endswith('.txt') for f in files):
            rel_path = os.path.relpath(root, BENCHMARKS_DIR)
            if rel_path != '.':
                groups.append(rel_path.replace('\\', '/'))
    return sorted(groups, key=lambda x: get_sort_val(x))

def run_benchmark(file_path, costs, max_cost):
    cmd = [EXE_PATH, str(file_path)] + [str(max_cost)] + [str(c) for c in costs]
    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        stdout, stderr = process.communicate()
        if process.returncode == 0:
            return parse_output(stdout)
        else:
            return {"RE": "error", "Cost": 0, "# REs": 0, "ICSize": 0, "Time (s)": 0.0, "Error": stderr}
    except Exception as e:
        return {"RE": "exception", "Cost": 0, "# REs": 0, "ICSize": 0, "Time (s)": 0.0, "Error": str(e)}

def calculate_statistics(df):
    success_df = df[df['RE'] != 'not_found']
    total = len(df)
    successful = len(success_df)
    failed = total - successful
    
    summary = pd.DataFrame({
        "Metric": ["Total Runs", "Successful Runs", "Failed Runs (not_found)"],
        "Value": [total, successful, failed]
    })
    
    if successful > 0:
        metrics = ['Cost', '# REs', 'ICSize', 'Time (s)']
        detailed_stats = success_df[metrics].describe(percentiles=[.25, .5, .75])
        return summary, detailed_stats
    else:
        return summary, None

groups = get_benchmark_groups()

selection_widget = widgets.SelectMultiple(
    options=groups,
    description='Groups:',
    disabled=False,
    layout={'width': '500px', 'height': '200px'}
)

limit_widget = widgets.BoundedIntText(
    value=50,
    min=1,
    max=10000,
    step=1,
    description='Limit:',
    disabled=False
)

run_all_checkbox = widgets.Checkbox(value=False, description='Run All Benchmarks', disabled=False, indent=False)
run_button = widgets.Button(description='Run Benchmarks', button_style='success', tooltip='Run selected benchmarks', icon='play')
output_area = widgets.Output()

def on_run_button_clicked(b):
    with output_area:
        clear_output()
        selected = list(selection_widget.value) if not run_all_checkbox.value else groups
        if not selected:
            print("No benchmark groups selected.")
            return
            
        limit = limit_widget.value
        print(f"Running benchmarks for: {', '.join(selected)} (Limit: {limit})")
        all_results = []
        count = 0
        
        for group in selected:
            if count >= limit: break
            group_path = os.path.join(BENCHMARKS_DIR, group)
            files = sorted([f for f in os.listdir(group_path) if f.endswith('.txt')], 
                           key=lambda x: get_sort_val(os.path.join(group, x)))
            
            for filename in files:
                if count >= limit: break
                file_path = os.path.join(group_path, filename)
                print(f"  [{count+1}/{limit}] Running {group}/{filename}...", end='\r')
                result = run_benchmark(file_path, DEFAULT_COSTS, MAX_COST)
                result['File'] = f"{group}/{filename}"
                all_results.append(result)
                count += 1
        
        clear_output()
        df = pd.DataFrame(all_results)
        df['sort_val'] = df['File'].apply(get_sort_val)
        df = df.sort_values(by='sort_val').drop(columns=['sort_val'])
        cols = ['File', 'RE', 'Cost', '# REs', 'ICSize', 'Time (s)']
        if 'Error' in df.columns: cols.append('Error')
        df = df[cols]
        
        print("### Benchmark Results")
        # Make table scrollable
        table_html = f"<div style='max-height: 400px; overflow-y: auto; border: 1px solid #ccc;'>{df.to_html(index=False)}</div>"
        display(HTML(table_html))
        
        summary, stats = calculate_statistics(df)
        print("\n### Summary Statistics")
        display(summary)
        
        if stats is not None:
            print("\n### Detailed Statistics (Successful Runs)")
            display(stats)
            
        # Comparison logic
        if os.path.exists('bottom_up_results.json'):
            import json
            with open('bottom_up_results.json', 'r') as fh:
                bu_data = json.load(fh)
            if isinstance(bu_data, list):
                bu_data = {r.get('benchmark_file', r.get('File', '')): r for r in bu_data}
            
            comparison_rows = []
            for _, row in df.iterrows():
                if row['RE'] not in ['not_found', 'error', 'exception']:
                    # match file
                    matched_key = None
                    for k in bu_data.keys():
                        if k.replace('\\', '/').endswith(row['File'].replace('\\', '/')):
                            matched_key = k
                            break
                    
                    if matched_key:
                        bu_res = bu_data[matched_key]
                        if bu_res.get('success') and bu_res.get('solutions'):
                            bu_best = min(bu_res['solutions'], key=lambda s: s.get('cost', 9999))
                            comp = {
                                'File': row['File'],
                                'BiDi_REs': row['# REs'],
                                'BU_REs': bu_best.get('all_res', bu_best.get('unique_res', 1)),
                                'BiDi_Time': row['Time (s)'],
                                'BU_Time': bu_best.get('running_time_s', 0.0),
                                'BiDi_Cost': row['Cost'],
                                'BU_Cost': bu_best.get('cost', row['Cost'])
                            }
                            # Ratio = BU / BiDi
                            comp['Ratio_REs'] = comp['BiDi_REs'] / comp['BU_REs'] if comp['BU_REs'] > 0 else np.nan
                            comp['Ratio_Time'] = comp['BiDi_Time'] / comp['BU_Time'] if comp['BU_Time'] > 0 else np.nan
                            comp['Ratio_Cost'] = comp['BiDi_Cost'] / comp['BU_Cost'] if comp['BU_Cost'] > 0 else np.nan
                            comparison_rows.append(comp)
            
            if comparison_rows:
                comp_df = pd.DataFrame(comparison_rows)
                print("\n### Comparison with Bottom-Up (Ratio = BU / BiDi)")
                comp_html = f"<div style='max-height: 400px; overflow-y: auto; border: 1px solid #ccc;'>{comp_df.to_html(index=False)}</div>"
                display(HTML(comp_html))
                
                print("\n### Ratio Statistics (BU / BiDi)")
                ratio_cols = ['Ratio_REs', 'Ratio_Time', 'Ratio_Cost']
                stats_df = comp_df[ratio_cols].describe(percentiles=[.25, .5, .75])
                display(stats_df)

run_button.on_click(on_run_button_clicked)
display(widgets.VBox([selection_widget, run_all_checkbox, limit_widget, run_button, output_area]))
